# Next bill prediction: data preprocessing

Run all cells from the repository root or `notebooks/`. This notebook validates the supplied synthetic panel, explores **training data only**, builds causal history features, and prepares model inputs. It does not train or select a predictive model or score the held-out test set.

Forecast at the close of month M using finalized records through M to predict M+1. See `docs/prediction_contract.md`. Raw data is never overwritten. Zero charges are valid; large bills are retained because they may be meaningful. Missing history is handled using training-only imputation and indicators.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/next_bill_amount_prediction_synthetic_dataset.csv").exists()), None)
assert ROOT is not None, "Run from the repository root or notebooks directory"
CSV = ROOT / "data/next_bill_amount_prediction_synthetic_dataset.csv"
EXPECTED_HASH = "1591046b1a25f22bb6da7f5ff18f84281afcfd62bbf3e7afd582607b15c8859f"
assert hashlib.sha256(CSV.read_bytes()).hexdigest() == EXPECTED_HASH
raw = pd.read_csv(CSV)
print("Raw shape:", raw.shape)

Raw shape: (17000, 15)


## 1. Validate before transforming

Fail explicitly on unexpected data rather than silently dropping records. Check exact schema, unique customer-month keys, month starts, finite nonnegative amounts, contiguous histories, and target alignment. Terminal targets cannot be verified and are kept outside the primary evaluation.

In [2]:
NUMERIC = ["data_usage_gb", "voice_calls", "sms_count", "addon_data_gb", "plan_fee",
           "discount_amount", "data_charge", "voice_charge", "addon_charge", "tax_amount", "bill_amount"]
assert set(raw.columns) == set(["customer_id", "billing_month", "plan_type", "next_bill_amount", *NUMERIC])
assert not raw.isna().any().any()
assert raw["plan_type"].isin(["Basic", "Standard", "Premium"]).all()
df = raw.copy()
df["billing_month"] = pd.to_datetime(df["billing_month"], format="%Y-%m-%d", errors="raise")
assert df["billing_month"].dt.is_month_start.all()
assert not df.duplicated(["customer_id", "billing_month"]).any()
for col in [*NUMERIC, "next_bill_amount"]:
    df[col] = pd.to_numeric(df[col], errors="raise")
assert np.isfinite(df[[*NUMERIC, "next_bill_amount"]].to_numpy()).all()
assert df[[*NUMERIC, "next_bill_amount"]].ge(0).all().all()
df = df.sort_values(["customer_id", "billing_month"]).reset_index(drop=True)
month_number = df["billing_month"].dt.year * 12 + df["billing_month"].dt.month
assert month_number.groupby(df["customer_id"]).diff().dropna().eq(1).all()
next_observed_bill = df.groupby("customer_id")["bill_amount"].shift(-1)
verified = next_observed_bill.notna()
assert np.allclose(df.loc[verified, "next_bill_amount"], next_observed_bill[verified], atol=0.01, rtol=0)
print(f"Verified labels: {verified.sum():,}; unverified terminal labels: {(~verified).sum():,}")

Verified labels: 16,000; unverified terminal labels: 1,000


## 2. Chronological partitions and training-only exploration

Train origins: Jan 2024-Feb 2025; validation: Mar 2025; test: Apr 2025. May 2025 is an optional forward set with unverified June labels. At the March close, February training labels have just become available. Do not fit this training set at the February close. Customer overlap is intentional: this task forecasts existing customers.

In [3]:
def split_name(frame):
    month = frame["billing_month"]
    result = pd.Series("forward", index=frame.index)
    result.loc[month <= pd.Timestamp("2025-02-01")] = "train"
    result.loc[month == pd.Timestamp("2025-03-01")] = "validation"
    result.loc[month == pd.Timestamp("2025-04-01")] = "test"
    return result

partitions = split_name(df)
assert partitions.value_counts().to_dict() == {"train": 14000, "validation": 1000, "test": 1000, "forward": 1000}
train_raw = df.loc[partitions.eq("train")]
display(train_raw.groupby("plan_type")[NUMERIC].mean().round(2))
display(train_raw[NUMERIC].describe().round(2))
display(df.assign(partition=partitions).groupby("partition")["billing_month"].agg(["min", "max", "size"]))

,data_usage_gb,voice_calls,sms_count,addon_data_gb,plan_fee,discount_amount,data_charge,voice_charge,addon_charge,tax_amount,bill_amount
plan_type,,,,,,,,,,,
Basic,14.68,254.14,89.48,1.02,199.0,15.03,145.81,6.21,10.21,62.31,408.51
Premium,15.52,245.56,92.47,1.03,699.0,14.76,0.00,5.73,10.27,126.04,826.28
Standard,15.03,247.52,89.94,1.04,399.0,14.94,12.37,5.66,10.39,74.25,486.73


,data_usage_gb,voice_calls,sms_count,addon_data_gb,plan_fee,discount_amount,data_charge,voice_charge,addon_charge,tax_amount,bill_amount
count,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00,14000.00
mean,14.97,249.90,90.20,1.03,369.30,14.95,65.05,5.90,10.29,78.41,514.00
std,8.28,77.25,40.48,2.57,176.95,23.06,117.76,13.55,25.72,29.75,195.05
min,1.00,0.00,0.00,0.00,199.00,0.00,0.00,0.00,0.00,22.32,146.32
25%,9.10,199.00,62.00,0.00,199.00,0.00,0.00,0.00,0.00,61.64,404.11
50%,13.42,251.00,90.00,0.00,399.00,0.00,0.00,0.00,0.00,71.82,470.82
75%,19.17,303.00,118.00,0.00,399.00,25.00,92.48,1.50,0.00,93.47,612.73
max,68.26,518.00,269.00,10.00,699.00,75.00,941.26,109.00,100.00,216.22,1417.47


,min,max,size
partition,,,
forward,2025-05-01,2025-05-01,1000
test,2025-04-01,2025-04-01,1000
train,2024-01-01,2025-02-01,14000
validation,2025-03-01,2025-03-01,1000


## 3. Features available at each forecast origin

History operations are grouped by customer and sorted by month. Lag 1 means the previous month; rolling windows include the current finalized month. No target enters feature construction. Early rows remain usable through imputation. Computing causal features before splitting allows validation and test origins to use legitimately observed earlier bills.

In [4]:
def build_features(frame):
    ordered = frame.sort_values(["customer_id", "billing_month"]).reset_index(drop=True)
    features = ordered[["plan_type", *NUMERIC]].copy()
    grouped = ordered.groupby("customer_id", sort=False)
    features["history_months"] = grouped.cumcount() + 1
    features["month_sin"] = np.sin(2 * np.pi * ordered["billing_month"].dt.month / 12)
    features["month_cos"] = np.cos(2 * np.pi * ordered["billing_month"].dt.month / 12)
    for col in ["bill_amount", "data_usage_gb", "voice_calls"]:
        for lag in [1, 2, 3]:
            features[f"{col}_lag{lag}"] = grouped[col].shift(lag)
        for window in [3, 6]:
            features[f"{col}_mean{window}"] = grouped[col].transform(
                lambda values: values.rolling(window, min_periods=1).mean())
        features[f"{col}_change"] = ordered[col] - features[f"{col}_lag1"]
    features["bill_std3"] = grouped["bill_amount"].transform(
        lambda values: values.rolling(3, min_periods=1).std(ddof=0))
    features["has_data_charge"] = ordered["data_charge"].gt(0).astype(int)
    features["has_voice_charge"] = ordered["voice_charge"].gt(0).astype(int)
    return features

X = build_features(df)
y = df["next_bill_amount"].copy()
assert not {"customer_id", "billing_month", "next_bill_amount"}.intersection(X.columns)
assert np.isfinite(X.select_dtypes("number").fillna(0).to_numpy()).all()
print("Feature matrix:", X.shape)

Feature matrix: (17000, 36)


## 4. Leakage checks

Recompute features from truncated histories and compare with the same origins in the full panel. Changing every target must also leave features unchanged. These checks catch future-history and target dependencies.

In [5]:
for cutoff in ["2024-01-01", "2024-06-01", "2025-03-01", "2025-04-01"]:
    mask = df["billing_month"].le(pd.Timestamp(cutoff))
    pd.testing.assert_frame_equal(build_features(df.loc[mask]), X.loc[mask].reset_index(drop=True))
changed_labels = df.copy()
changed_labels["next_bill_amount"] = -999
pd.testing.assert_frame_equal(build_features(changed_labels), X)
first_rows = df.groupby("customer_id").head(1).index
assert X.loc[first_rows, "bill_amount_lag1"].isna().all()
print("Temporal, target independence, and customer-boundary checks passed")

Temporal, target independence, and customer-boundary checks passed


## 5. Fit preprocessing on training data only

Median imputation with missingness indicators handles short histories; standardization supports scale-sensitive future models. Plans receive one-hot encoding; unseen plans at inference are handled without failure. Transforming held-out features does not fit statistics or evaluate their labels. Keep this fitted transformer with any future model; do not fit it again at prediction time.

In [6]:
numeric_features = X.select_dtypes("number").columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("plan", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["plan_type"]),
], remainder="drop", verbose_feature_names_out=True)

masks = {name: partitions.eq(name) for name in ["train", "validation", "test", "forward"]}
prepared = {"train": preprocessor.fit_transform(X.loc[masks["train"]])}
for name in ["validation", "test", "forward"]:
    prepared[name] = preprocessor.transform(X.loc[masks[name]])
feature_names = preprocessor.get_feature_names_out()
for name, matrix in prepared.items():
    assert matrix.shape == (int(masks[name].sum()), len(feature_names))
    assert np.isfinite(matrix).all()
    print(name, matrix.shape)
train_target_months = df.loc[masks["train"], "billing_month"] + pd.offsets.MonthBegin(1)
assert train_target_months.max() <= df.loc[masks["validation"], "billing_month"].min()
assert hashlib.sha256(CSV.read_bytes()).hexdigest() == EXPECTED_HASH

train (14000, 50)
validation (1000, 50)
test (1000, 50)
forward (1000, 50)


## 6. Optional exports and next modeling stage

Set `EXPORT = True` to save matrices, aligned labels, customer-month metadata, feature names, and the fitted transformer under ignored `data/processed/`. Only load joblib artifacts you trust. The default run creates no processed files.

Next: compare previous-bill and trailing-mean baselines with a regression model using validation MAE/RMSE/WAPE. Lock feature/model choices before a single primary test evaluation. The existing audit baseline scores used all verified rows, including holdouts; treat those scores as descriptive, not independent validation. May-origin labels remain unverified and are not part of the primary test.

In [7]:
EXPORT = False
if EXPORT:
    import joblib
    output = ROOT / "data/processed"
    output.mkdir(parents=True, exist_ok=True)
    for name, matrix in prepared.items():
        np.savez_compressed(output / f"{name}.npz", X=matrix, y=y.loc[masks[name]].to_numpy())
        metadata = df.loc[masks[name], ["customer_id", "billing_month"]].copy()
        metadata["target_month"] = metadata["billing_month"] + pd.offsets.MonthBegin(1)
        metadata.to_csv(output / f"{name}_metadata.csv", index=False)
    (output / "feature_names.json").write_text(json.dumps(feature_names.tolist(), indent=2), encoding="utf-8")
    joblib.dump(preprocessor, output / "preprocessor.joblib")
    print("Saved to", output.relative_to(ROOT))